# Umba Take- Assessment – Part A: Real-Time Fraud Detection
## Objective
The objective of this project is to build a machine learning pipeline that detects fraudulent financial transactions in real time. 
The pipeline includes data loading, data validation, feature engineering, model development, evaluation using time-based validation, and generation of fraud probabilities for unseen transactions.

## Data Loading
This section imports the required libraries and loads the transaction, identity, and sample submission datasets. 
It also performs an initial inspection of the data and checks the target class distribution.

In [5]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")

# Path to the data folder
DATA_DIR = r"C:\Users\Moses\Downloads\umba-fraud-takehome (1)\candidate\data"

# Load datasets
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
identity = pd.read_csv(os.path.join(DATA_DIR, "identity.csv"))
sample_submission = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print("="*60)
print("DATASET SHAPES")
print("="*60)

print(f"Train: {train.shape}")
print(f"Test: {test.shape}")
print(f"Identity: {identity.shape}")
print(f"Sample Submission: {sample_submission.shape}")

print("\nFraud Distribution")
print(train["isFraud"].value_counts())

print(f"\nFraud Rate: {train['isFraud'].mean()*100:.2f}%")

DATASET SHAPES
Train: (120000, 61)
Test: (40000, 60)
Identity: (47140, 14)
Sample Submission: (40000, 2)

Fraud Distribution
isFraud
0    115874
1      4126
Name: count, dtype: int64

Fraud Rate: 3.44%


## 2. Data Integrity Checks

Before building any model, the datasets are checked for duplicate records, missing target values, unique transaction identifiers, and consistency between the training and test datasets. These checks ensure the data is suitable for analysis.

In [6]:
print("="*70)
print("DATA INTEGRITY CHECKS")
print("="*70)

# Check duplicate TransactionIDs
print("\nDuplicate TransactionIDs")
print(f"Train: {train['TransactionID'].duplicated().sum()}")
print(f"Test: {test['TransactionID'].duplicated().sum()}")
print(f"Identity: {identity['TransactionID'].duplicated().sum()}")

# Missing target values
print("\nMissing values in target")
print(train['isFraud'].isnull().sum())

# Compare train and test columns
train_cols = set(train.columns) - {'isFraud'}
test_cols = set(test.columns)

print("\nTrain and Test have identical feature columns?")
print(train_cols == test_cols)

# Check if TransactionID is unique
print("\nTransactionID uniqueness")
print("Train:", train['TransactionID'].is_unique)
print("Test :", test['TransactionID'].is_unique)

# Number of unique TransactionIDs in identity
print("\nIdentity Summary")
print("Total rows:", len(identity))
print("Unique TransactionIDs:", identity['TransactionID'].nunique())

DATA INTEGRITY CHECKS

Duplicate TransactionIDs
Train: 0
Test: 0
Identity: 5530

Missing values in target
0

Train and Test have identical feature columns?
True

TransactionID uniqueness
Train: True
Test : True

Identity Summary
Total rows: 47140
Unique TransactionIDs: 41610


## 3. Data Leakage Detection
Some variables may contain information that would not be available when making real-time predictions. This section identifies such features to prevent information leakage that could artificially inflate model performance.

In [7]:
print("=" * 70)
print("POTENTIAL DATA LEAKAGE")
print("=" * 70)

# Check if the leakage column exists
if "flagged_for_review" in train.columns:
    print("\n'flagged_for_review' exists in the dataset.")
    
    print("\nValue Counts:")
    print(train["flagged_for_review"].value_counts(dropna=False))
    
    print("\nFraud Rate by flagged_for_review:")
    print(train.groupby("flagged_for_review")["isFraud"].mean())
    
else:
    print("'flagged_for_review' not found.")

POTENTIAL DATA LEAKAGE

'flagged_for_review' exists in the dataset.

Value Counts:
flagged_for_review
0.0    111488
1.0      8512
Name: count, dtype: int64

Fraud Rate by flagged_for_review:
flagged_for_review
0.0    0.002682
1.0    0.449601
Name: isFraud, dtype: float64


## 4. Identity Data Analysis
The identity dataset is analysed separately because a transaction may have multiple associated identity records. Understanding its structure is necessary before joining it with the transaction data.

In [8]:
print("=" * 70)
print("IDENTITY DATA ANALYSIS")
print("=" * 70)

# Count how many identity rows each transaction has
identity_counts = identity.groupby("TransactionID").size()

print("\nNumber of identity records per TransactionID:")
print(identity_counts.value_counts().sort_index())

print("\nSummary statistics:")
print(identity_counts.describe())

print("\nMaximum number of identity records for a single TransactionID:")
print(identity_counts.max())

print("\nExample TransactionIDs with multiple identity records:")
display(identity_counts[identity_counts > 1].head(10))

IDENTITY DATA ANALYSIS

Number of identity records per TransactionID:
1    36080
2     5530
Name: count, dtype: int64

Summary statistics:
count    41610.000000
mean         1.132901
std          0.339472
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          2.000000
dtype: float64

Maximum number of identity records for a single TransactionID:
2

Example TransactionIDs with multiple identity records:


TransactionID
1000009    2
1000026    2
1000052    2
1000086    2
1000145    2
1000201    2
1000265    2
1000268    2
1000474    2
1000486    2
dtype: int64

## 4. Missing Value Analysis

This section identifies features with missing values and measures the percentage of missing data in each feature. Understanding the missing data pattern helps determine appropriate preprocessing strategies for model training.

In [9]:
print("=" * 70)
print("MISSING VALUE ANALYSIS")
print("=" * 70)

# Percentage of missing values
missing = train.isnull().mean() * 100

missing = (
    missing[missing > 0]
    .sort_values(ascending=False)
    .reset_index()
)

missing.columns = ["Feature", "Missing (%)"]

print(f"\nFeatures with missing values: {len(missing)}")

display(missing)

print("\nTop 20 Features with the Highest Missing Percentage")
display(missing.head(20))

MISSING VALUE ANALYSIS

Features with missing values: 37


,Feature,Missing (%)
0,dist2,72.825833
1,V5,65.238333
2,V10,65.108333
3,V20,64.921667
4,V15,64.910000
5,D2,55.031667
6,D5,52.117500
7,D4,47.934167
8,dist1,42.041667
9,M5,40.027500



Top 20 Features with the Highest Missing Percentage


,Feature,Missing (%)
0,dist2,72.825833
1,V5,65.238333
2,V10,65.108333
3,V20,64.921667
4,V15,64.910000
5,D2,55.031667
6,D5,52.117500
7,D4,47.934167
8,dist1,42.041667
9,M5,40.027500


## 5. Feature Data Types
This section examines the data types of all variables and identifies numerical and categorical features. This information is used to design the preprocessing pipeline, including missing value imputation and categorical encoding.

In [10]:
print("=" * 70)
print("FEATURE DATA TYPES")
print("=" * 70)

# Count each datatype
print("\nData Type Counts:")
print(train.dtypes.value_counts())

# List categorical columns
categorical_cols = train.select_dtypes(include=["object"]).columns.tolist()

# List numerical columns
numerical_cols = train.select_dtypes(include=["int64", "float64"]).columns.tolist()

print(f"\nNumber of Numerical Features: {len(numerical_cols)}")
print(f"Number of Categorical Features: {len(categorical_cols)}")

print("\nCategorical Features:")
print(categorical_cols)

FEATURE DATA TYPES

Data Type Counts:
float64    34
int64      14
object     13
Name: count, dtype: int64

Number of Numerical Features: 48
Number of Categorical Features: 13

Categorical Features:
['country', 'currency', 'channel', 'card_type', 'card_bank', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6']


## 6. Preparing the Datasets

The training and testing datasets are prepared for modelling. The original datasets are copied to preserve the raw data, the leakage feature (`flagged_for_review`) is removed, the target variable (`isFraud`) is separated from the predictors, and the remaining features are prepared for subsequent preprocessing and model training.

In [11]:
print("=" * 70)
print("PREPARING THE DATASETS")
print("=" * 70)

# Create copies
train_df = train.copy()
test_df = test.copy()

# Remove leakage feature
train_df = train_df.drop(columns=["flagged_for_review"])
test_df = test_df.drop(columns=["flagged_for_review"])

# Separate target
y = train_df["isFraud"]

# Remove target from training features
train_df = train_df.drop(columns=["isFraud"])

print("Training Features Shape :", train_df.shape)
print("Training Target Shape   :", y.shape)
print("Test Features Shape     :", test_df.shape)

print("\nLeakage feature removed successfully.")

PREPARING THE DATASETS
Training Features Shape : (120000, 59)
Training Target Shape   : (120000,)
Test Features Shape     : (40000, 59)

Leakage feature removed successfully.


## 7. Inspecting the Identity Dataset

The identity dataset is explored to understand its structure before it is merged with the transaction data. This step examines the dataset dimensions, data types, and the percentage of missing values in each identity feature. The analysis helps identify potential data quality issues and guides the strategy for handling identity information during feature engineering.

In [12]:
print("=" * 70)
print("IDENTITY DATASET OVERVIEW")
print("=" * 70)

print("\nShape:", identity.shape)

print("\nData Types:")
print(identity.dtypes)

print("\nMissing Values (%):")
identity_missing = (
    identity.isnull().mean() * 100
).sort_values(ascending=False)

display(identity_missing.to_frame("Missing (%)"))

IDENTITY DATASET OVERVIEW

Shape: (47140, 14)

Data Types:
TransactionID      int64
DeviceType        object
DeviceInfo        object
id_01            float64
id_02            float64
id_03            float64
id_04            float64
id_05            float64
id_06            float64
id_07            float64
id_08            float64
id_09            float64
id_10            float64
id_11            float64
dtype: object

Missing Values (%):


,Missing (%)
id_11,53.165040
id_10,49.974544
id_09,46.979211
id_08,43.773865
id_07,40.992787
id_06,37.781078
id_05,35.086975
id_04,31.694951
id_03,28.992363
id_02,25.793381


## 8. Aggregating the Identity Dataset

The identity dataset contains multiple records for some TransactionIDs, while the transaction dataset contains only one record per transaction. To prevent duplicate rows during merging, the identity data is aggregated so that each TransactionID has a single representation. Numeric variables are summarized using statistical measures, categorical variables are represented by their most frequent value, and the number of identity records per transaction is retained as an additional feature. This preserves useful information while ensuring a one-to-one relationship between the transaction and identity datasets.

In [13]:
print("=" * 70)
print("AGGREGATING IDENTITY DATA")
print("=" * 70)

# Count the number of identity records per transaction
identity["identity_record_count"] = (
    identity.groupby("TransactionID")["TransactionID"]
            .transform("count")
)

# Separate numeric and categorical columns
numeric_cols = identity.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_cols.remove("TransactionID")

categorical_cols = identity.select_dtypes(include=["object"]).columns.tolist()

# Aggregation rules
agg_dict = {}

# Numerical features -> mean
for col in numeric_cols:
    agg_dict[col] = "mean"

# Categorical features -> first non-null value
for col in categorical_cols:
    agg_dict[col] = "first"

# Aggregate
identity_agg = (
    identity
    .groupby("TransactionID", as_index=False)
    .agg(agg_dict)
)

print("Original Identity Shape :", identity.shape)
print("Aggregated Identity Shape:", identity_agg.shape)

display(identity_agg.head())

AGGREGATING IDENTITY DATA
Original Identity Shape : (47140, 15)
Aggregated Identity Shape: (41610, 15)


,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,id_10,id_11,identity_record_count,DeviceType,DeviceInfo
0,1000009,-1.12105,NaN,-0.39025,-2.86375,-1.10265,2.33945,NaN,NaN,NaN,-1.5629,1.6867,2.0,mobile,samsung
1,1000020,0.85000,1.8176,NaN,2.31650,NaN,0.27700,NaN,NaN,NaN,NaN,NaN,1.0,desktop,Infinix
2,1000026,-1.46730,NaN,NaN,NaN,1.15000,2.49255,NaN,0.89655,NaN,NaN,0.6120,2.0,mobile,samsung
3,1000032,-0.36510,-3.0375,1.29850,0.34460,NaN,1.02850,NaN,-0.05660,NaN,NaN,-0.1818,1.0,mobile,SM-G973F
4,1000033,-1.89430,-1.1976,-0.65450,3.09540,-0.50540,-0.58610,-0.6285,-0.20690,NaN,NaN,NaN,1.0,mobile,SM-G973F


## 9. Exploring the Time Variable

The transaction time variable is examined to understand the chronological order of the data. The minimum and maximum transaction times are compared between the training and test datasets to verify that the test data occurs after the training data. This confirms that a time-based validation strategy is appropriate and helps prevent data leakage by preserving the natural order of transactions.

In [14]:
print("=" * 70)
print("TRANSACTION TIME ANALYSIS")
print("=" * 70)

print("\nTraining Data")
print("Minimum TransactionDT:", train_df["TransactionDT"].min())
print("Maximum TransactionDT:", train_df["TransactionDT"].max())

print("\nTest Data")
print("Minimum TransactionDT:", test_df["TransactionDT"].min())
print("Maximum TransactionDT:", test_df["TransactionDT"].max())

print("\nDoes the test data occur after the training data?")
print(
    test_df["TransactionDT"].min() >
    train_df["TransactionDT"].max()
)

TRANSACTION TIME ANALYSIS

Training Data
Minimum TransactionDT: 13
Maximum TransactionDT: 11657391

Test Data
Minimum TransactionDT: 11657574
Maximum TransactionDT: 15551902

Does the test data occur after the training data?
True


## 10. Merging the Transaction and Identity Datasets

After aggregating the identity dataset, it is merged with both the training and test transaction datasets using TransactionID as the common key. A left join is used to ensure that all transactions are retained even when corresponding identity information is unavailable. This creates a single dataset that combines transaction features with the available identity information for model development.

In [15]:
print("=" * 70)
print("MERGING TRANSACTION AND IDENTITY DATA")
print("=" * 70)

# Merge aggregated identity data
train_df = train_df.merge(
    identity_agg,
    on="TransactionID",
    how="left"
)

test_df = test_df.merge(
    identity_agg,
    on="TransactionID",
    how="left"
)

print("Training Shape After Merge :", train_df.shape)
print("Test Shape After Merge     :", test_df.shape)

print("\nChecking duplicate TransactionIDs after merge")

print("Train duplicates:",
      train_df["TransactionID"].duplicated().sum())

print("Test duplicates:",
      test_df["TransactionID"].duplicated().sum())

print("\nTransactions with identity information")

print("Train:",
      train_df["DeviceType"].notna().sum())

print("Test:",
      test_df["DeviceType"].notna().sum())

MERGING TRANSACTION AND IDENTITY DATA
Training Shape After Merge : (120000, 73)
Test Shape After Merge     : (40000, 73)

Checking duplicate TransactionIDs after merge
Train duplicates: 0
Test duplicates: 0

Transactions with identity information
Train: 31216
Test: 10394


## 11. Feature Engineering

Several new features are created to improve the predictive power of the model. These include transformations of transaction amounts, time-based features extracted from the transaction timestamp, indicators describing whether the purchaser and recipient email domains match, and behavioural features derived from previous transaction amounts. These engineered variables provide additional information that may help distinguish fraudulent transactions from legitimate ones.

In [16]:
print("=" * 70)
print("FEATURE ENGINEERING")
print("=" * 70)

# -------------------------------
# Transaction Amount Features
# -------------------------------
train_df["LogTransactionAmt"] = np.log1p(train_df["TransactionAmt"])
test_df["LogTransactionAmt"] = np.log1p(test_df["TransactionAmt"])

# -------------------------------
# Time Features
# -------------------------------
SECONDS_PER_DAY = 24 * 60 * 60

train_df["TransactionDay"] = train_df["TransactionDT"] // SECONDS_PER_DAY
test_df["TransactionDay"] = test_df["TransactionDT"] // SECONDS_PER_DAY

train_df["TransactionHour"] = (train_df["TransactionDT"] // 3600) % 24
test_df["TransactionHour"] = (test_df["TransactionDT"] // 3600) % 24

# -------------------------------
# Email Match Feature
# -------------------------------
train_df["EmailMatch"] = (
    train_df["P_emaildomain"] ==
    train_df["R_emaildomain"]
).astype(int)

test_df["EmailMatch"] = (
    test_df["P_emaildomain"] ==
    test_df["R_emaildomain"]
).astype(int)

# -------------------------------
# Currency Normalized Amount
# -------------------------------
train_df["AmtPerPreviousTxn"] = (
    train_df["TransactionAmt"] /
    (train_df["sender_prev_txn_count"] + 1)
)

test_df["AmtPerPreviousTxn"] = (
    test_df["TransactionAmt"] /
    (test_df["sender_prev_txn_count"] + 1)
)

print("Feature engineering completed.")

print("\nNew Shape (Train):", train_df.shape)
print("New Shape (Test):", test_df.shape)

print("\nNew Features Added:")
print([
    "LogTransactionAmt",
    "TransactionDay",
    "TransactionHour",
    "EmailMatch",
    "AmtPerPreviousTxn"
])

FEATURE ENGINEERING
Feature engineering completed.

New Shape (Train): (120000, 78)
New Shape (Test): (40000, 78)

New Features Added:
['LogTransactionAmt', 'TransactionDay', 'TransactionHour', 'EmailMatch', 'AmtPerPreviousTxn']


## 12. Creating the Time-Based Training and Validation Split

The training dataset is sorted chronologically using the transaction timestamp before being divided into training and validation sets. Instead of randomly splitting the data, the earliest 80% of transactions are used for training while the most recent 20% are reserved for validation. This approach simulates a real-world fraud detection scenario, where models are trained on historical transactions and evaluated on future transactions, reducing the risk of temporal data leakage.

In [17]:
from sklearn.model_selection import train_test_split

# Sort by transaction time
train_df = train_df.sort_values("TransactionDT").reset_index(drop=True)
y = y.loc[train_df.index]

# Use the last 20% of transactions for validation
split_index = int(len(train_df) * 0.80)

X_train = train_df.iloc[:split_index].copy()
X_valid = train_df.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_valid = y.iloc[split_index:].copy()

print("=" * 70)
print("TIME-BASED TRAIN / VALIDATION SPLIT")
print("=" * 70)

print("Training Set :", X_train.shape)
print("Validation Set:", X_valid.shape)

print("\nTraining Time Range")
print(X_train["TransactionDT"].min(), "to", X_train["TransactionDT"].max())

print("\nValidation Time Range")
print(X_valid["TransactionDT"].min(), "to", X_valid["TransactionDT"].max())

TIME-BASED TRAIN / VALIDATION SPLIT
Training Set : (96000, 78)
Validation Set: (24000, 78)

Training Time Range
13 to 9358843

Validation Time Range
9358947 to 11657391


## 13. Building the Preprocessing Pipeline

A preprocessing pipeline is created to ensure that the data is prepared consistently before model training. The TransactionID column is removed because it is only an identifier and does not provide predictive information. Numerical features are imputed using the median to handle missing values, while categorical features are imputed using the most frequent category and encoded into numerical values. Combining these preprocessing steps into a single pipeline ensures that the same transformations are applied to both the training and test datasets, reducing errors and improving reproducibility.

In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report

print("=" * 70)
print("BUILDING PREPROCESSING PIPELINE")
print("=" * 70)

# Remove TransactionID from modelling features
drop_cols = ["TransactionID"]

X_train_model = X_train.drop(columns=drop_cols)
X_valid_model = X_valid.drop(columns=drop_cols)

# Identify column types
categorical_features = X_train_model.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X_train_model.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

# Numeric preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

# Full preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("\nPreprocessing pipeline created successfully.")

BUILDING PREPROCESSING PIPELINE
Numeric features: 62
Categorical features: 15

Preprocessing pipeline created successfully.


## 14. Training the Baseline Random Forest Model

A Random Forest classifier is trained as the baseline machine learning model for fraud detection. The preprocessing pipeline is combined with the classifier so that data preparation and model training are performed within a single workflow. Class weighting is used to reduce the impact of class imbalance, and the trained model is evaluated on the validation dataset using PR-AUC and ROC-AUC metrics. The baseline model provides a reference point for comparing more advanced models.

In [19]:
print("=" * 70)
print("TRAINING BASELINE RANDOM FOREST MODEL")
print("=" * 70)

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train_model, y_train)

valid_probs_rf = rf_model.predict_proba(X_valid_model)[:, 1]

pr_auc_rf = average_precision_score(y_valid, valid_probs_rf)
roc_auc_rf = roc_auc_score(y_valid, valid_probs_rf)

print("Random Forest Validation PR-AUC:", round(pr_auc_rf, 4))
print("Random Forest Validation ROC-AUC:", round(roc_auc_rf, 4))

TRAINING BASELINE RANDOM FOREST MODEL
Random Forest Validation PR-AUC: 0.17
Random Forest Validation ROC-AUC: 0.7852


## 15. Setting Up the LightGBM Library

LightGBM is introduced as a gradient boosting framework designed for efficient and accurate classification on large datasets. Before training the model, the library is imported and its installation is verified. This step ensures that the required package is available for the subsequent model development.

In [22]:
import lightgbm as lgb

print("LightGBM version:", lgb.__version__)

LightGBM version: 4.6.0


## 16. Training the Improved LightGBM Model

An improved LightGBM classifier is trained using the same preprocessing pipeline as the baseline model. To address the severe class imbalance in the fraud detection dataset, the positive class weight is calculated automatically and supplied to the classifier. The model is then evaluated on the validation dataset using PR-AUC and ROC-AUC, allowing its performance to be compared directly with the baseline Random Forest model.

In [25]:
print("=" * 70)
print("TRAINING IMPROVED LIGHTGBM MODEL")
print("=" * 70)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

lgbm_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", lgb.LGBMClassifier(
        objective="binary",
        n_estimators=1200,
        learning_rate=0.02,
        num_leaves=16,
        max_depth=6,
        min_child_samples=80,
        subsample=0.75,
        colsample_bytree=0.75,
        reg_alpha=0.5,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ))
])

lgbm_model.fit(X_train_model, y_train)

valid_probs_lgbm = lgbm_model.predict_proba(X_valid_model)[:, 1]

pr_auc_lgbm = average_precision_score(y_valid, valid_probs_lgbm)
roc_auc_lgbm = roc_auc_score(y_valid, valid_probs_lgbm)

print("Improved LightGBM Validation PR-AUC:", round(pr_auc_lgbm, 4))
print("Improved LightGBM Validation ROC-AUC:", round(roc_auc_lgbm, 4))

TRAINING IMPROVED LIGHTGBM MODEL
Improved LightGBM Validation PR-AUC: 0.1787
Improved LightGBM Validation ROC-AUC: 0.7854


## 17. Optimizing the Classification Threshold

The default probability threshold of 0.50 does not always produce the best balance between precision and recall, particularly for highly imbalanced fraud detection problems. A precision-recall curve is therefore used to evaluate different probability thresholds. The F1-score is calculated for each threshold, and the threshold that maximizes the F1-score is selected. This optimized threshold improves the balance between correctly identifying fraudulent transactions and reducing false alarms.

In [26]:
from sklearn.metrics import precision_recall_curve
import numpy as np
import pandas as pd

precision, recall, thresholds = precision_recall_curve(
    y_valid,
    valid_probs_lgbm
)

# Compute F1 score
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)

best_index = np.argmax(f1_scores)

best_threshold = thresholds[best_index]

print("=" * 70)
print("THRESHOLD OPTIMIZATION")
print("=" * 70)

print("Best Threshold:", round(best_threshold, 4))
print("Precision:", round(precision[best_index], 4))
print("Recall:", round(recall[best_index], 4))
print("F1 Score:", round(f1_scores[best_index], 4))

THRESHOLD OPTIMIZATION
Best Threshold: 0.7201
Precision: 0.1934
Recall: 0.3512
F1 Score: 0.2494


## 18. Retraining the Final Model on the Complete Training Dataset

After selecting the best model and identifying the optimal probability threshold, the final LightGBM model is retrained using the entire training dataset. Unlike the earlier stage, where a portion of the data was reserved for validation, this step allows the model to learn from all available labelled transactions. The preprocessing pipeline and optimized model parameters are retained to produce the final fraud detection model that will be used to generate predictions for the unseen test dataset.

In [28]:
print("=" * 70)
print("TRAINING FINAL MODEL ON FULL TRAINING DATA")
print("=" * 70)

# Use the full prepared training data, but remove TransactionID
X_full_model = train_df.drop(columns=["TransactionID"]).copy()

# y already contains the target labels
scale_pos_weight_full = (y == 0).sum() / (y == 1).sum()

final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", lgb.LGBMClassifier(
        objective="binary",
        n_estimators=1200,
        learning_rate=0.02,
        num_leaves=16,
        max_depth=6,
        min_child_samples=80,
        subsample=0.75,
        colsample_bytree=0.75,
        reg_alpha=0.5,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight_full,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ))
])

final_model.fit(X_full_model, y)

print("Final model trained successfully on the full training dataset.")

TRAINING FINAL MODEL ON FULL TRAINING DATA
Final model trained successfully on the full training dataset.


## 19. Generating Test Predictions

After training the final LightGBM model, predictions are generated for every transaction in the unseen test dataset. Since the objective is to estimate the probability that each transaction is fraudulent, the model outputs the predicted fraud probability instead of a binary class label. The resulting dataframe contains the TransactionID together with the corresponding predicted fraud probability (`isFraud_prob`). This output will be saved as the final submission file.

In [29]:
print("=" * 70)
print("GENERATING TEST PREDICTIONS")
print("=" * 70)

X_test_model = test_df.drop(columns=["TransactionID"]).copy()

test_probs = final_model.predict_proba(X_test_model)[:, 1]

predictions = pd.DataFrame({
    "TransactionID": test_df["TransactionID"],
    "isFraud_prob": test_probs
})

print(predictions.head())
print("\nShape:", predictions.shape)
print("\nProbability range:", predictions["isFraud_prob"].min(), "to", predictions["isFraud_prob"].max())

GENERATING TEST PREDICTIONS
   TransactionID  isFraud_prob
0        1120000      0.277446
1        1120001      0.434202
2        1120002      0.719660
3        1120003      0.367119
4        1120004      0.077598

Shape: (40000, 2)

Probability range: 0.002717632483962948 to 0.9724626673030082


## 20. Saving the Prediction File

The generated fraud probability predictions are saved as a CSV file in the required submission format. Saving the predictions ensures that the results can be submitted for evaluation without rerunning the notebook. As a final verification step, the saved file is reloaded to confirm that it has been written correctly and contains the expected number of rows and columns.

In [30]:
output_path = os.path.join(DATA_DIR, "predictions.csv")

predictions.to_csv(output_path, index=False)

print("predictions.csv saved successfully.")
print("Saved to:", output_path)

# Quick check
saved_predictions = pd.read_csv(output_path)
print(saved_predictions.head())
print(saved_predictions.shape)

predictions.csv saved successfully.
Saved to: C:\Users\Moses\Downloads\umba-fraud-takehome (1)\candidate\data\predictions.csv
   TransactionID  isFraud_prob
0        1120000      0.277446
1        1120001      0.434202
2        1120002      0.719660
3        1120003      0.367119
4        1120004      0.077598
(40000, 2)


## 21. Saving the Final Trained Model

The final fraud detection pipeline is saved using Joblib to preserve both the preprocessing steps and the trained LightGBM model. Saving the complete pipeline allows the model to be reloaded later for prediction, evaluation, or deployment without repeating the entire training process. The saved model can therefore be used directly on new transaction data while applying the same preprocessing steps learned during training.

In [31]:
import joblib
import os

# Save the trained model
model_path = os.path.join(DATA_DIR, "final_model.pkl")
joblib.dump(final_model, model_path)

print("Model saved successfully!")
print("Location:", model_path)

Model saved successfully!
Location: C:\Users\Moses\Downloads\umba-fraud-takehome (1)\candidate\data\final_model.pkl


In [33]:
import json
import numpy as np

# Replace NaN with None so json.dumps emits valid `null`
sample_api_input_clean = sample_api_input.where(pd.notnull(sample_api_input), None)

sample_json = sample_api_input_clean.to_dict(orient="records")

# This is the string you paste into Swagger — NOT the raw Python object
sample_json_str = json.dumps(sample_json, indent=2)
print(sample_json_str)

[
  {
    "TransactionID": 1120000,
    "TransactionDT": 11657574,
    "TransactionAmt": 143.9,
    "country": "KE",
    "currency": "KES",
    "channel": "mobile_money",
    "card_type": "debit",
    "card_bank": "opay",
    "card1": 7236,
    "card2": 316.0,
    "card3": 150.0,
    "card5": 220.0,
    "addr1": 322.0,
    "addr2": 254.0,
    "dist1": NaN,
    "dist2": 45.62,
    "P_emaildomain": "gmail.com",
    "R_emaildomain": "gmail.com",
    "recipient_account_age_days": 241,
    "sender_prev_txn_count": 23,
    "C1": 4,
    "C2": 4,
    "C3": 3,
    "C4": 0,
    "C5": 2,
    "C6": 1,
    "C7": 0,
    "C8": 0,
    "D1": 366.3,
    "D2": NaN,
    "D3": 10.4,
    "D4": NaN,
    "D5": 67.9,
    "M1": "F",
    "M2": null,
    "M3": "T",
    "M4": null,
    "M5": "T",
    "M6": "T",
    "V1": -1.0874,
    "V2": 2.1544,
    "V3": -0.0533,
    "V4": 1.0086,
    "V5": 1.2348,
    "V6": -1.4606,
    "V7": NaN,
    "V8": NaN,
    "V9": NaN,
    "V10": NaN,
    "V11": 2.5023,
    "V12": 0.15

In [34]:
sample_api_input_2 = test_df.drop(columns=["TransactionID"]).head(2).copy()
sample_api_input_2.insert(0, "TransactionID", test_df["TransactionID"].iloc[:2].values)

sample_api_input_2_clean = sample_api_input_2.where(pd.notnull(sample_api_input_2), None)
sample_json_2 = sample_api_input_2_clean.to_dict(orient="records")

import json
print(json.dumps(sample_json_2, indent=2))

[
  {
    "TransactionID": 1120000,
    "TransactionDT": 11657574,
    "TransactionAmt": 143.9,
    "country": "KE",
    "currency": "KES",
    "channel": "mobile_money",
    "card_type": "debit",
    "card_bank": "opay",
    "card1": 7236,
    "card2": 316.0,
    "card3": 150.0,
    "card5": 220.0,
    "addr1": 322.0,
    "addr2": 254.0,
    "dist1": NaN,
    "dist2": 45.62,
    "P_emaildomain": "gmail.com",
    "R_emaildomain": "gmail.com",
    "recipient_account_age_days": 241,
    "sender_prev_txn_count": 23,
    "C1": 4,
    "C2": 4,
    "C3": 3,
    "C4": 0,
    "C5": 2,
    "C6": 1,
    "C7": 0,
    "C8": 0,
    "D1": 366.3,
    "D2": NaN,
    "D3": 10.4,
    "D4": NaN,
    "D5": 67.9,
    "M1": "F",
    "M2": null,
    "M3": "T",
    "M4": null,
    "M5": "T",
    "M6": "T",
    "V1": -1.0874,
    "V2": 2.1544,
    "V3": -0.0533,
    "V4": 1.0086,
    "V5": 1.2348,
    "V6": -1.4606,
    "V7": NaN,
    "V8": NaN,
    "V9": NaN,
    "V10": NaN,
    "V11": 2.5023,
    "V12": 0.15